# HRNet-W18 RGB-D multitask — локальное обучение (Windows + RTX 4060 8 GB)

Целевая машина: **AMD Ryzen 5 5600G + RTX 4060 (8 GB) + 16 GB RAM, Windows 10/11**.

Что делает этот ноутбук:
1. Проверяет GPU и зависимости.
2. Тянет архив датасета **с того же Я.Диска**, что и `notebooks/colab_train.ipynb` (`https://disk.yandex.ru/d/Je56nUcC9hiHFQ`, папка `/Jacquard_V2`, 12 zip-файлов). Можно вместо этого положить архивы вручную в `ZIP_DIR`.
3. Распаковывает архивы (параллельно, удаляет zip после успешной распаковки — чтобы 65 + 65 GB не лопнули диск).
4. Строит object-wise split (или переиспользует существующий `splits/jacquard_v2.json`).
5. **Smoke-test:** один forward+backward с реальным батчем — меряет `step_time`, `gpu_mem_peak`, `dataload_fraction` и оценивает **ETA до конца обучения**. Если на `batch=2 image_size=512` будет OOM или ETA окажется неприемлемым — здесь же можно прервать и подкрутить `BATCH_SIZE` / `IMAGE_SIZE`.
6. Запускает 150-эпохное обучение с конфигом `configs/multitask_local512.yaml` (`rgbd_multitask`, 512×512, batch=2, accum=8, AMP, чекпоинт раз в 5 эпох, `iter_log.jsonl` каждый шаг).
7. Рисует `metrics.csv` и `iter_log.jsonl` (опционально — после/во время обучения).

**Расшифровка всех метрик из логов** — в [`docs/training_metrics.md`](../docs/training_metrics.md).

**Структура данных:**
* `DATA_ROOT/zips/` — временные zip-архивы (можно удалить после распаковки).
* `DATA_ROOT/Jacquard_V2/<object_id>/...` — собственно датасет (~70 GB).
* `RUNS_ROOT/multitask_local512/` — `best.pth`, `last.pth`, `epoch_*.pth`, `metrics.csv`, `iter_log.jsonl`, `train.log`.

## 0. Пути и параметры (отредактируй ячейку под себя)

Все пути — Windows-стиль с `r"..."` или прямыми слэшами (Python всё съест).

In [ ]:
import os
from pathlib import Path

# === ПУТИ ===
# Корень репозитория: ноутбук должен лежать в repo/notebooks/, поэтому ../
REPO_DIR = Path(os.path.abspath('..')).resolve()

# Куда положить датасет (zip + распакованные объекты).
# Желательно SSD/NVMe; нужно ~140 GB свободного места на момент распаковки
# (после удаления zip остаётся ~70 GB).
DATA_ROOT = Path(r'D:/Datasets/JacquardV2')
ZIP_DIR   = DATA_ROOT / 'zips'
DATA_DIR  = DATA_ROOT / 'Jacquard_V2'

# Куда складывать чекпоинты, метрики, логи. Можно вынести на другой диск.
RUNS_ROOT = REPO_DIR / 'train_results'
RUN_NAME  = 'hrnet_w18_rgbd_multitask_local512'
SAVE_DIR  = RUNS_ROOT / RUN_NAME

# === ИСТОЧНИК ДАТАСЕТА ===
# Я.Диск с 12 архивами. Если zip-файлы уже лежат в ZIP_DIR — скачивание пропустится.
# Если уже распаковано в DATA_DIR — пропустятся и скачивание, и распаковка.
YANDEX_DISK_FOLDER_URL = 'https://disk.yandex.ru/d/Je56nUcC9hiHFQ'
YANDEX_FOLDER_PATH     = '/Jacquard_V2'
MAX_PARALLEL_DOWNLOADS = 4
MAX_PARALLEL_UNZIP     = 2  # 16 GB RAM — не больше 2 потоков чтобы не лопнуть кэш
DELETE_ZIPS_AFTER_EXTRACT = True

# === ОБУЧЕНИЕ ===
# Базовый конфиг — на нём построены остальные значения.
CONFIG_FILE = 'configs/multitask_local512.yaml'
# Любой leaf-параметр конфига можно переопределить через OVERRIDES; это удобнее,
# чем редактировать YAML.
IMAGE_SIZE  = 512
BATCH_SIZE  = 2     # для 8 GB VRAM @ 512px с AMP
ACCUM_STEPS = 8     # эффективный батч = 16
NUM_WORKERS = 4     # 16 GB RAM держит 4 worker'а с persistent_workers
EPOCHS      = 150
SAVE_EVERY_N_EPOCHS = 5  # epoch_004.pth, epoch_009.pth, ... + всегда last.pth + best.pth

# Resume автоматически с last.pth, если он уже есть в SAVE_DIR.
AUTO_RESUME = True

# Splits: один раз посчитать, потом переиспользовать.
SPLITS_PATH = REPO_DIR / 'splits' / 'jacquard_v2.json'

# Sanity-print
for label, p in [('REPO_DIR', REPO_DIR), ('DATA_ROOT', DATA_ROOT),
                 ('ZIP_DIR', ZIP_DIR), ('DATA_DIR', DATA_DIR),
                 ('SAVE_DIR', SAVE_DIR), ('SPLITS_PATH', SPLITS_PATH)]:
    print(f'{label:12} = {p}')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)
(REPO_DIR / 'splits').mkdir(parents=True, exist_ok=True)

## 1. Проверка GPU + Python окружения

Если `nvidia-smi` упало — драйвер NVIDIA не видит RTX 4060. Проверь Device Manager и переустанови драйвер.

Если `torch.cuda.is_available()` вернёт `False` при работающем `nvidia-smi` — у тебя CPU-only torch. Переустанови:
```
pip uninstall torch torchvision
pip install --index-url https://download.pytorch.org/whl/cu124 torch torchvision
```

In [ ]:
!nvidia-smi

In [ ]:
import sys, platform
import torch
print('python  :', sys.version.split()[0], '/', platform.system(), platform.release())
print('torch   :', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device  :', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f'vram    : {free/1e9:.2f} GB free / {total/1e9:.2f} GB total')

## 2. Установка зависимостей (только если ещё не стоят)

Запускай эту ячейку **только при первом запуске**. На последующих запусках пропускай — лишний `pip install` может перетянуть transitive deps и перезапустить kernel.

In [ ]:
# Проверяем, что критичные пакеты импортируются. Если хоть один падает — раскомментируй
# pip-строку ниже и запусти эту ячейку.
missing = []
for pkg in ['torch', 'torchvision', 'timm', 'cv2', 'numpy', 'tifffile',
            'matplotlib', 'pandas', 'yaml', 'tqdm', 'requests', 'imagecodecs',
            'pynvml']:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)
print('missing:', missing)

# ↓ если есть пропуски — раскомментируй и запусти. На Windows + Python 3.11/3.12 это
# обычная установка из PyPI; cu124 wheels работают на RTX 4060.
# !pip install --index-url https://download.pytorch.org/whl/cu124 torch torchvision
# !pip install timm opencv-python tifffile matplotlib pandas pyyaml tqdm requests imagecodecs pynvml

## 3. Скачивание датасета с Я.Диска

Идемпотентно: уже скачанные/распакованные файлы пропускаются.

**Если ты уже скачал zip-архивы вручную** — положи их в `ZIP_DIR` (`D:/Datasets/JacquardV2/zips/` по умолчанию) и эта ячейка просто увидит, что всё на месте.

**Если ты уже распаковал датасет** в `DATA_DIR` — переходи к ячейке **5** (Splits).

In [ ]:
import os, time, threading, glob
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

API = 'https://cloud-api.yandex.net/v1/disk/public/resources'

# Если в DATA_DIR уже > 1000 объектных папок — считаем, что датасет распакован,
# и пропускаем и скачивание, и распаковку.
_object_dirs = [d for d in os.listdir(DATA_DIR)
                if os.path.isdir(os.path.join(DATA_DIR, d))]
DATASET_ALREADY_EXTRACTED = len(_object_dirs) > 1000
print(f'DATA_DIR содержит {len(_object_dirs)} подпапок → '
      f'{"уже распакован, пропускаем" if DATASET_ALREADY_EXTRACTED else "нужно скачать/распаковать"}')

_print_lock = threading.Lock()
def _log(msg):
    with _print_lock:
        print(msg, flush=True)

def _list_public_folder(public_url, sub_path='', limit=100):
    offset = 0
    while True:
        params = {'public_key': public_url, 'limit': limit, 'offset': offset}
        if sub_path:
            params['path'] = sub_path
        r = requests.get(API, params=params, timeout=60)
        r.raise_for_status()
        items = r.json().get('_embedded', {}).get('items', [])
        if not items:
            return
        for it in items:
            if it['type'] == 'file':
                yield it['name'], it.get('path')
        if len(items) < limit:
            return
        offset += limit

def _download_one(public_url, dest, inner_path):
    name = os.path.basename(dest)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        _log(f'  skip (exists): {name}')
        return dest
    r = requests.get(API + '/download',
                     params={'public_key': public_url, 'path': inner_path},
                     timeout=60)
    r.raise_for_status()
    href = r.json()['href']
    _log(f'  start: {name}')
    t0 = time.time()
    tmp = dest + '.part'
    with requests.get(href, stream=True, timeout=600) as resp:
        resp.raise_for_status()
        with open(tmp, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
    os.replace(tmp, dest)
    size_gb = os.path.getsize(dest) / 1e9
    dt = time.time() - t0
    _log(f'  done:  {name}  ({size_gb:.2f} GB, '
         f'{size_gb*1000/max(dt,1e-3):.1f} MB/s, {dt:.0f}s)')
    return dest

if not DATASET_ALREADY_EXTRACTED:
    files = list(_list_public_folder(YANDEX_DISK_FOLDER_URL, YANDEX_FOLDER_PATH))
    print(f'found {len(files)} file(s) in {YANDEX_FOLDER_PATH or "<root>"}')
    print(f'downloading with {MAX_PARALLEL_DOWNLOADS} parallel workers into {ZIP_DIR}')
    t_start = time.time()
    errors = []
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_DOWNLOADS) as pool:
        futures = {
            pool.submit(_download_one, YANDEX_DISK_FOLDER_URL,
                        str(ZIP_DIR / name), inner_path): name
            for name, inner_path in files
        }
        for fut in as_completed(futures):
            try:
                fut.result()
            except Exception as e:
                errors.append((futures[fut], e))
                _log(f'  FAILED: {futures[fut]}: {e}')
    if errors:
        raise RuntimeError(f'{len(errors)} download(s) failed: '
                           + ', '.join(n for n, _ in errors))
    print(f'all {len(files)} zips ready in {ZIP_DIR} '
          f'(total {time.time()-t_start:.0f}s)')

# Что сейчас лежит в ZIP_DIR
print('---')
for z in sorted(glob.glob(str(ZIP_DIR / '*.zip'))):
    print(f'  {os.path.basename(z):40s} {os.path.getsize(z)/1e9:.2f} GB')

## 4. Распаковка zip → плоская структура `DATA_DIR/<object_id>/...`

Каждый zip содержит верхний уровень `Jacquard*Dataset_<N>/` который мы выпрямляем после извлечения, чтобы `tools/prepare_split.py` нашёл объекты на одном уровне.

Если `DELETE_ZIPS_AFTER_EXTRACT=True` (по умолчанию) — каждый успешно распакованный zip удаляется сразу же, чтобы пиковое заполнение диска оставалось ниже 70 + 65 = 135 GB.

In [ ]:
import zipfile, shutil

if DATASET_ALREADY_EXTRACTED:
    print('skip: dataset already extracted in', DATA_DIR)
else:
    def _extract_one(zip_path):
        name = os.path.basename(zip_path)
        _log(f'  start: {name}')
        t0 = time.time()
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)
        suffix = ''
        if DELETE_ZIPS_AFTER_EXTRACT:
            os.remove(zip_path)
            suffix = ' [zip deleted]'
        _log(f'  done:  {name}  ({time.time()-t0:.0f}s){suffix}')

    zips = sorted(glob.glob(str(ZIP_DIR / '*.zip')))
    if not zips:
        print('no zips left in', ZIP_DIR, '— assuming already extracted')
    else:
        print(f'extracting {len(zips)} zip(s) with {MAX_PARALLEL_UNZIP} parallel workers')
        t_start = time.time()
        errors = []
        with ThreadPoolExecutor(max_workers=MAX_PARALLEL_UNZIP) as pool:
            futures = {pool.submit(_extract_one, z): z for z in zips}
            for fut in as_completed(futures):
                try:
                    fut.result()
                except Exception as e:
                    errors.append((futures[fut], e))
                    _log(f'  FAILED: {os.path.basename(futures[fut])}: {e}')
        if errors:
            raise RuntimeError(f'{len(errors)} extract(s) failed: '
                               + ', '.join(os.path.basename(n) for n, _ in errors))
        print(f'all extracted in {time.time()-t_start:.0f}s')

    # Flatten: each zip extracts into a Jacquard*_Dataset_<N>/ folder. Move
    # all object subfolders one level up so DATA_DIR is flat.
    for shard_dir in (sorted(glob.glob(str(DATA_DIR / 'JacquardV2_Dataset_*')))
                      + sorted(glob.glob(str(DATA_DIR / 'Jacquard_Dataset_*')))):
        if not os.path.isdir(shard_dir):
            continue
        for entry in os.listdir(shard_dir):
            src = os.path.join(shard_dir, entry)
            dst = os.path.join(DATA_DIR, entry)
            if os.path.exists(dst):
                continue
            shutil.move(src, dst)
        try:
            os.rmdir(shard_dir)
        except OSError:
            pass

n = sum(1 for d in os.listdir(DATA_DIR)
        if os.path.isdir(os.path.join(DATA_DIR, d)))
print(f'object folders in {DATA_DIR}: {n}  (expected ≈ 11619)')

## 5. Object-wise split (10/10/80 val/test/train, seed=0)

Считается один раз и сохраняется в `splits/jacquard_v2.json`. На последующих запусках переиспользуется — это гарантирует, что val/test остаются прежними между ранами и метрики сравнимы.

In [ ]:
import subprocess, json

if SPLITS_PATH.exists():
    print('split exists, reusing:', SPLITS_PATH)
else:
    cmd = [sys.executable, str(REPO_DIR / 'tools' / 'prepare_split.py'),
           '--root', str(DATA_DIR),
           '--out',  str(SPLITS_PATH),
           '--val-frac', '0.1',
           '--test-frac', '0.1',
           '--seed', '0']
    print('running:', ' '.join(cmd))
    subprocess.check_call(cmd, cwd=str(REPO_DIR))

split = json.load(open(SPLITS_PATH))
print(f'train={len(split["train"])} val={len(split["val"])} '
      f'test={len(split["test"])}  (sum={sum(len(v) for v in split.values())})')

## 6. Smoke-test: 1 батч + замер времени + ETA до конца обучения

Эта ячейка — **критическая**. Она проверяет, что:
1. Конфиг + модель собираются без ошибок.
2. На `BATCH_SIZE` × `IMAGE_SIZE` всё помещается в VRAM (нет OOM).
3. Forward+backward+optimizer идёт за разумное время.

В конце печатает **оценку времени** до конца 150-эпохного рана — ты решаешь, стоит ли запускать.

Если выпадает CUDA OOM — сократи `BATCH_SIZE` до 1 (тогда подними `ACCUM_STEPS` до 16, чтобы эффективный батч остался 16) или уменьши `IMAGE_SIZE` до 448.

In [ ]:
# Добавляем repo-root в sys.path, чтобы импортировать grasp_seg.* из ноутбука.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import yaml, time as _time
import torch.nn as nn
from torch.utils.data import DataLoader

from grasp_seg.data import (AugConfig, DatasetConfig, JacquardV2GraspSeg,
                            collate_fn, load_split)
from grasp_seg.losses import MultiTaskGraspLoss
from grasp_seg.models import build_model

# Загружаем конфиг и применяем CLI-style overrides из ячейки 0.
with open(REPO_DIR / CONFIG_FILE) as f:
    cfg = yaml.safe_load(f)
cfg['dataset']['root']        = str(DATA_DIR)
cfg['dataset']['splits_path'] = str(SPLITS_PATH)
cfg['dataset']['image_size']  = IMAGE_SIZE
cfg['dataset']['num_workers'] = NUM_WORKERS
cfg['trainer']['batch_size']  = BATCH_SIZE
cfg['trainer']['accum_steps'] = ACCUM_STEPS
cfg['trainer']['epochs']      = EPOCHS
cfg['trainer']['save_dir']    = str(SAVE_DIR)
cfg['trainer']['save_every_n_epochs'] = SAVE_EVERY_N_EPOCHS

# Один батч из train-loader'а. DatasetConfig содержит только parametry
# препроцессинга (image_size / input_mode / mask_mode / ...) — пути берёт
# JacquardV2GraspSeg из списка grasp-файлов, а не из конфига.
ds_cfg = DatasetConfig(
    image_size=cfg['dataset']['image_size'],
    input_mode=cfg['dataset']['input_mode'],
    mask_mode=cfg['dataset']['mask_mode'],
    num_angle_bins=cfg['dataset']['num_angle_bins'],
    length_scale=cfg['dataset']['length_scale'],
    use_stereo_depth=cfg['dataset']['use_stereo_depth'],
)
aug_cfg = AugConfig(**{k: v for k, v in cfg['augmentation'].items()})

split = load_split(cfg['dataset']['splits_path'])
train_ds = JacquardV2GraspSeg(split.train[:64], ds_cfg, aug=aug_cfg, is_training=True)
loader   = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, collate_fn=collate_fn, pin_memory=True)

in_ch = {'rgb': 3, 'depth': 1, 'rgbd': 4}[cfg['dataset']['input_mode']]
model_cfg = {
    'mask_mode':      cfg['dataset']['mask_mode'],
    'backbone':       cfg['model']['backbone'],
    'in_channels':    in_ch,
    'pretrained':     cfg['model']['pretrained'],
    'num_angle_bins': cfg['dataset']['num_angle_bins'],
}
model = build_model(model_cfg).cuda()
loss_fn = MultiTaskGraspLoss(
    pos_weight=cfg['loss']['pos_weight'],
    cos_weight=cfg['loss']['cos_weight'],
    sin_weight=cfg['loss']['sin_weight'],
    width_weight=cfg['loss']['width_weight'],
    bce_pos_weight=cfg['loss']['bce_pos_weight'],
).cuda()
opt    = torch.optim.SGD(model.parameters(), lr=cfg['trainer']['lr'],
                         momentum=cfg['trainer']['momentum'])
scaler = torch.amp.GradScaler('cuda', enabled=True)

torch.cuda.reset_peak_memory_stats()
model.train()
batch = next(iter(loader))
x = batch['input'].cuda(non_blocking=True)
y = {k: v.cuda(non_blocking=True) for k, v in batch['target'].items()}

# 3 шага для прогрева JIT/cuDNN, потом 5 шагов для замера.
for _ in range(3):
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', enabled=True):
        pred = model(x)
        loss = loss_fn(pred, y)['loss']
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update()
torch.cuda.synchronize()
t0 = _time.perf_counter()
for _ in range(5):
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', enabled=True):
        pred = model(x)
        loss = loss_fn(pred, y)['loss']
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update()
torch.cuda.synchronize()
step_time = (_time.perf_counter() - t0) / 5
peak_gb   = torch.cuda.max_memory_allocated() / (1024**3)
free_gb   = torch.cuda.mem_get_info()[0] / 1e9

# ETA = epoch_time × epochs.
# epoch_time ≈ steps/epoch × accum × step_time + 1 эпоха валидации (~3% сверху).
n_train = len(split.train)
steps_per_epoch = max(n_train // (BATCH_SIZE * ACCUM_STEPS), 1)
epoch_s = steps_per_epoch * ACCUM_STEPS * step_time * 1.03
eta_h   = epoch_s * EPOCHS / 3600

print(f'\n=== smoke-test results ===')
print(f'image_size            = {IMAGE_SIZE}px')
print(f'batch_size            = {BATCH_SIZE}  (effective = {BATCH_SIZE*ACCUM_STEPS} via accum={ACCUM_STEPS})')
print(f'step_time (1 micro)   = {step_time*1000:.1f} ms')
print(f'gpu_mem_peak          = {peak_gb:.2f} GB')
print(f'gpu_mem_free_after    = {free_gb:.2f} GB')
print(f'steps/epoch           = {steps_per_epoch}')
print(f'estimated epoch time  = {epoch_s/60:.1f} min')
print(f'estimated total ETA   = {eta_h:.1f} h  ({eta_h/24:.1f} days) for {EPOCHS} epochs')

del model, loss_fn, opt, scaler, batch, x, y, pred, loss
torch.cuda.empty_cache()

## 7. Запуск обучения

Запускается через `tools/train.py` как subprocess — это даёт чистый процесс под обучение и позволяет логам идти прямо в ноутбук **и** в файл `train.log` под `SAVE_DIR`. Subprocess можно прервать кнопкой ⏹ — последняя строка metrics.csv уже будет сохранена, и `last.pth` тоже.

Resume: если `last.pth` существует — `tools/train.py --resume` подхватит его и продолжит с (epoch+1).

In [ ]:
import subprocess, sys

RESUME_PATH = SAVE_DIR / 'last.pth'
resume_args = ['--resume', str(RESUME_PATH)] if (AUTO_RESUME and RESUME_PATH.exists()) else []

cmd = [sys.executable, str(REPO_DIR / 'tools' / 'train.py'),
       '--config', CONFIG_FILE,
       *resume_args,
       f'dataset.root={DATA_DIR}',
       f'dataset.splits_path={SPLITS_PATH}',
       f'dataset.image_size={IMAGE_SIZE}',
       f'dataset.num_workers={NUM_WORKERS}',
       'dataset.persistent_workers=true',
       'dataset.prefetch_factor=2',
       f'trainer.epochs={EPOCHS}',
       f'trainer.batch_size={BATCH_SIZE}',
       f'trainer.accum_steps={ACCUM_STEPS}',
       f'trainer.save_dir={SAVE_DIR}',
       'trainer.save_every_epoch=false',
       f'trainer.save_every_n_epochs={SAVE_EVERY_N_EPOCHS}',
       'trainer.iter_log_path=iter_log.jsonl']
print('command:')
print(' '.join(repr(c) if ' ' in str(c) else str(c) for c in cmd))
print()

# Дублируем stdout в файл train.log под SAVE_DIR.
log_path = SAVE_DIR / 'train.log'
with open(log_path, 'a', encoding='utf-8') as logf:
    proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, bufsize=1, text=True,
                            encoding='utf-8', errors='replace')
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            logf.write(line)
            logf.flush()
    finally:
        ret = proc.wait()
print(f'\n[train.py exited with code {ret}]')

## 8. Графики из `metrics.csv` (per-epoch)

Запусти **по ходу обучения** (в новой kernel-вкладке) или после, чтобы посмотреть кривые. `metrics.csv` обновляется после каждой эпохи, так что её можно читать на лету.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = SAVE_DIR / 'metrics.csv'
df = pd.read_csv(csv_path)
print(f'rows: {len(df)}, columns: {list(df.columns)}')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
# (1) train loss + components
for col in [c for c in df.columns if c.startswith('train_')
            and c.split('_', 1)[1] in ('loss', 'pos_bce', 'pos_dice',
                                        'cos', 'sin', 'width')]:
    axes[0].plot(df['epoch'], df[col], label=col)
axes[0].set_title('train loss components'); axes[0].legend(); axes[0].grid()
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
# (2) val IoU/Dice
for col in ['val_miou', 'val_miou_fg', 'val_dice', 'val_dice_fg']:
    if col in df.columns:
        axes[1].plot(df['epoch'], df[col], label=col)
axes[1].set_title('val IoU / Dice'); axes[1].legend(); axes[1].grid()
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('score')
# (3) val angle MAE и cos/sin MSE (multitask)
for col in ['val_ang_mae_deg']:
    if col in df.columns:
        ax2 = axes[2]
        ax2.plot(df['epoch'], df[col], 'r-', label=col)
        ax2.set_ylabel('deg', color='r')
for col in ['val_cos_mse', 'val_sin_mse']:
    if col in df.columns:
        axes[2].plot(df['epoch'], df[col], '--', label=col, alpha=0.6)
axes[2].set_title('val angle metrics'); axes[2].legend(); axes[2].grid()
axes[2].set_xlabel('epoch')
plt.tight_layout(); plt.show()

## 9. График из `iter_log.jsonl` (per-step)

Если включён `iter_log_path` (по умолчанию — да), в `SAVE_DIR/iter_log.jsonl` лежит по строке на каждый optimizer-шаг. Это даёт **гладкую кривую loss** внутри одной эпохи (а не одну точку как в `metrics.csv`).

In [ ]:
import json

iter_path = SAVE_DIR / 'iter_log.jsonl'
rows = [json.loads(l) for l in open(iter_path, encoding='utf-8') if l.strip()]
print(f'iter_log rows: {len(rows)}  ({rows[-1]["epoch"] - rows[0]["epoch"] + 1} epochs covered)')
df_iter = pd.DataFrame(rows)

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
for col in ['loss', 'pos_bce', 'pos_dice', 'cos', 'sin', 'width']:
    if col in df_iter.columns:
        # rolling mean чтобы не было шума
        ax.plot(df_iter['step'], df_iter[col].rolling(50, min_periods=1).mean(), label=col)
ax.set_title('per-step training loss (rolling mean over 50 steps)')
ax.set_xlabel('global step'); ax.set_ylabel('loss')
ax.legend(); ax.grid()
plt.tight_layout(); plt.show()